In [1]:
import requests
import os

class WebHDFSClient:
    def __init__(self, namenode_host, namenode_port=9870, user="jovyan"):
        self.base_url = f"http://{namenode_host}:{namenode_port}/webhdfs/v1"
        self.user = user

    def upload_file(self, local_path, hdfs_path, overwrite=True):
        """
        Uploads a file using the WebHDFS two-step protocol.
        Step 1: Submit a CREATE request to NameNode. It returns a redirect to a DataNode.
        Step 2: Upload the actual data to the DataNode.
        """
        # Construct the operation URL
        # op=CREATE is the WebHDFS operation for uploading files
        params = {
            "op": "CREATE",
            "overwrite": "true" if overwrite else "false",
            "user.name": self.user
        }
        
        # Step 1: Request to create file (returns 307 Redirect)
        # We set allow_redirects=False because we need to capture the DataNode URL first
        create_url = f"{self.base_url}{hdfs_path}"
        print(f"Step 1: contacting NameNode at {create_url}...")
        
        response = requests.put(create_url, params=params, allow_redirects=False)
        
        if response.status_code != 307:
            raise Exception(f"Failed to init upload. Status: {response.status_code}, Msg: {response.text}")
        
        # The NameNode gives us the URL of the specific DataNode where data should go
        datanode_url = response.headers['Location']
        print(f"Step 2: Redirected to DataNode. Uploading data...")

        # Step 2: Put the actual file content to the DataNode
        with open(local_path, 'rb') as file_data:
            # We must explicitly pass the content-type for binary transfer
            headers = {'Content-Type': 'application/octet-stream'}
            upload_response = requests.put(datanode_url, data=file_data, headers=headers)

        if 200 <= upload_response.status_code < 300:
            print(f"SUCCESS: File uploaded to {hdfs_path}")
            print(f"Location: {upload_response.headers.get('Location', 'N/A')}")
        else:
            raise Exception(f"DataNode write failed. Status: {upload_response.status_code}")

    def list_status(self, hdfs_path):
        """
        Lists files in a directory using WebHDFS.
        """
        params = {"op": "LISTSTATUS", "user.name": self.user}
        list_url = f"{self.base_url}{hdfs_path}"
        
        response = requests.get(list_url, params=params)
        
        if response.status_code == 200:
            data = response.json()
            files = data.get("FileStatuses", {}).get("FileStatus", [])
            print(f"\n--- Contents of {hdfs_path} ---")
            for f in files:
                print(f"Name: {f['pathSuffix']} | Size: {f['length']} bytes | Owner: {f['owner']}")
        else:
            print(f"Failed to list directory. Status: {response.status_code}")



In [2]:
hdfs = WebHDFSClient(namenode_host="namenode", namenode_port=9870)

local_filename = "alice_in_wonderland.txt"
local_path = f"/home/jovyan/shared/{local_filename}"


target_dir = "/data"
target_path = f"{target_dir}/{local_filename}"

try:
    # Ensure the target directory works (WebHDFS creates parent dirs automatically on CREATE?)
    # Actually, for safety, standard 'hdfs dfs -put' logic implies we might need mkdirs, 
    # but WebHDFS 'CREATE' often handles the file creation. 
    # NOTE: If the parent dir doesn't exist, WebHDFS might fail depending on config.
    # Let's assume our previous script created /user/test_data, or we can add a mkdirs call.
    
    hdfs.upload_file(local_path, target_path)
    
    # 4. Verify
    hdfs.list_status(target_dir)
    
except Exception as e:
    print(f"An error occurred: {e}")

Step 1: contacting NameNode at http://namenode:9870/webhdfs/v1/data/alice_in_wonderland.txt...
Step 2: Redirected to DataNode. Uploading data...
SUCCESS: File uploaded to /data/alice_in_wonderland.txt
Location: hdfs://namenode:8020/data/alice_in_wonderland.txt

--- Contents of /data ---
Name: alice_in_wonderland.txt | Size: 148208 bytes | Owner: jovyan
